In [2]:
from pathlib import Path
import pandas as pd

# ---- the ONE thing to edit: your project root ----
BASE = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")

# auto-pick the most recent raw population download (so there's no date to edit)
raw_path = sorted((BASE / "raw" / "population").glob("nomis_population_raw_*.csv"))[-1]
print("Using raw file:", raw_path.name)

raw = pd.read_csv(raw_path, encoding="utf-8-sig")
raw.columns = [c.strip().upper() for c in raw.columns]
spine = pd.read_csv(BASE / "clean" / "geography_spine.csv")
pop   = pd.read_csv(BASE / "clean" / "population_itl2.csv")

missing = ["E08000016", "E08000019"]

# 1. What ARE these two places?
print(raw.loc[raw.GEOGRAPHY_CODE.isin(missing), ["GEOGRAPHY_CODE", "GEOGRAPHY_NAME"]].drop_duplicates())

# 2. Are they in the spine under these exact codes?
print("Found in spine:", spine.la_code.isin(missing).sum(), "of 2")

# 3. All-ages sanity check: total population in the latest year
latest = pop.year.max()
print("Latest year:", latest, "| total population:", f"{pop.loc[pop.year == latest, 'population'].sum():,.0f}")

Using raw file: nomis_population_raw_2026-08-06.csv
   GEOGRAPHY_CODE GEOGRAPHY_NAME
53      E08000016       Barnsley
56      E08000019      Sheffield
Found in spine: 0 of 2
Latest year: 2025 | total population: 66,856,852


In [3]:
import re

# load the raw LAD -> ITL lookup the spine was built from
geo_files = [p for p in (BASE / "raw" / "geography").glob("*.csv") if "ITL" in p.name or "Lookup" in p.name]
lk = pd.read_csv(sorted(geo_files)[-1], encoding="utf-8-sig")
lk.columns = [c.strip() for c in lk.columns]

def col(pat): return [c for c in lk.columns if re.fullmatch(pat, c, re.I)]
lad_cd, lad_nm = col(r"LAD\d*CD")[0], col(r"LAD\d*NM")[0]
itl2cd, itl2nm = col(r"ITL2\d*CD")[0], col(r"ITL2\d*NM")[0]

# 1. Are Barnsley/Sheffield in the RAW lookup by their Nomis codes?
print("--- by code ---")
print(lk.loc[lk[lad_cd].isin(["E08000016","E08000019"]), [lad_cd,lad_nm,itl2cd,itl2nm]].to_string(index=False))

# 2. Are they in the lookup under their NAMES (maybe with different codes)?
print("\n--- by name ---")
print(lk.loc[lk[lad_nm].str.contains("Barnsley|Sheffield", case=False, na=False), [lad_cd,lad_nm,itl2cd,itl2nm]].to_string(index=False))

# 3. What South Yorkshire codes ARE in the lookup? (E08000016-19 = the four boroughs)
print("\n--- all four South Yorkshire boroughs in lookup ---")
print(lk.loc[lk[lad_cd].isin(["E08000016","E08000017","E08000018","E08000019"]), [lad_cd,lad_nm,itl2cd,itl2nm]].to_string(index=False))

--- by code ---
Empty DataFrame
Columns: [LAD25CD, LAD25NM, ITL225CD, ITL225NM]
Index: []

--- by name ---
  LAD25CD   LAD25NM ITL225CD        ITL225NM
E08000039 Sheffield     TLE3 South Yorkshire
E08000038  Barnsley     TLE3 South Yorkshire

--- all four South Yorkshire boroughs in lookup ---
  LAD25CD   LAD25NM ITL225CD        ITL225NM
E08000018 Rotherham     TLE3 South Yorkshire
E08000017 Doncaster     TLE3 South Yorkshire


In [4]:
from pathlib import Path
import pandas as pd

BASE = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")

def read_robust(path, **kw):
    for enc in ("cp1252", "latin-1"):
        try: return pd.read_csv(path, encoding=enc, dtype=str, **kw)
        except UnicodeDecodeError: continue

chg = read_robust(next((BASE / "raw" / "geography").rglob("Changes.csv")))
chg.columns = [c.strip().upper() for c in chg.columns]

codes = ["E08000016", "E08000038", "E08000019", "E08000039"]
cols = [c for c in ["GEOGCD","GEOGNM","GEOGCD_P","GEOGNM_P","YEAR"] if c in chg.columns]

print("Rows touching Barnsley/Sheffield codes:")
print(chg[chg.GEOGCD.isin(codes) | chg.GEOGCD_P.isin(codes)][cols].to_string(index=False))

print("\nDoes E08000038 (Barnsley's new code) appear as a predecessor?")
sub = chg[chg.GEOGCD_P == "E08000038"][cols]
print(sub.to_string(index=False) if len(sub) else "  (no)")

# how widespread is ambiguity? predecessors that point to more than one successor
multi = chg.groupby("GEOGCD_P")["GEOGCD"].nunique()
print("\nPredecessors mapping to >1 successor:", int((multi > 1).sum()))
print(multi[multi > 1].head(20).to_string())

Rows touching Barnsley/Sheffield codes:
   GEOGCD    GEOGNM  GEOGCD_P  GEOGNM_P YEAR
E08000016  Barnsley      00CC  Barnsley 2009
E08000019 Sheffield      00CG Sheffield 2009
E08000038  Barnsley E08000016  Barnsley 2025
E08000039 Sheffield E08000016  Barnsley 2025
E08000039 Sheffield E08000019 Sheffield 2025

Does E08000038 (Barnsley's new code) appear as a predecessor?
  (no)

Predecessors mapping to >1 successor: 19126
GEOGCD_P
00        6
001       2
002       3
003       3
004       2
005       2
006       3
007       2
008       2
009       3
00GG01    2
00GG02    3
00GG03    2
00GG05    3
00GG06    2
00GG08    3
00GG10    2
00GG11    3
00GG12    5
00GG13    2
